In [1]:
###############################################################################
# 01 - Build Master Sample Table
#
# Purpose:
#   Build the standardized Master Sample Table used by all downstream analyses.
#
# Inputs:
#   - Sponsor metadata files for GSE146680, GSE203655, and GSE212277
#   - Salmon quant.sf directory structure
#   - metadata_helpers.R
#
# Output:
#   - results/metadata/master_sample_table.csv
#
# Design:
#   Original metadata formats remain unchanged.
#   Dataset-specific helper functions transform each file into the same schema.
###############################################################################

In [2]:
# Personal Library and Packages

user_lib <- path.expand("~/R/x86_64-pc-linux-gnu-library/4.4")

if (dir.exists(user_lib)) {
  .libPaths(unique(c(user_lib, .libPaths())))
}

required_packages <- c(
  "readxl",
  "dplyr",
  "stringr"
)

for (pkg in required_packages) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    stop("Missing required package: ", pkg)
  }

  suppressPackageStartupMessages(
    library(pkg, character.only = TRUE)
  )
}

cat("Required packages loaded successfully.\n")

Required packages loaded successfully.


In [3]:
# Project Paths

base_dir <- "/standard/harrislab/capstone_2026"
project_dir <- path.expand("~/MSDS/capstone_project")

scripts_dir <- file.path(project_dir, "scripts")
metadata_results_dir <- file.path(project_dir, "results", "metadata")

dir.create(
  metadata_results_dir,
  recursive = TRUE,
  showWarnings = FALSE
)

cat("Sponsor data:", base_dir, "\n")
cat("Project directory:", project_dir, "\n")

Sponsor data: /standard/harrislab/capstone_2026 
Project directory: /home/kvq4nvn/MSDS/capstone_project 


In [4]:
# Load metadata helper functions

metadata_helper_path <- file.path(
  scripts_dir,
  "metadata_helpers.R"
)

if (!file.exists(metadata_helper_path)) {
  stop("metadata_helpers.R was not found at: ", metadata_helper_path)
}

source(metadata_helper_path)

cat("metadata_helpers.R loaded successfully.\n")

metadata_helpers.R loaded successfully.


In [5]:
# Build the Sample Inventory

find_quant_files <- function(dataset_id) {

  dataset_dir <- file.path(
    base_dir,
    "quants",
    dataset_id
  )

  sample_dirs <- list.dirs(
    dataset_dir,
    recursive = FALSE,
    full.names = TRUE
  )

  quant_paths <- file.path(
    sample_dirs,
    "quant.sf"
  )

  sample_names <- sub(
    "_quant$",
    "",
    basename(sample_dirs)
  )

  data.frame(
    dataset = dataset_id,
    sample = sample_names,
    quant_path = quant_paths,
    file_exists = file.exists(quant_paths),
    stringsAsFactors = FALSE
  )
}

datasets <- c(
  "GSE146680",
  "GSE203655",
  "GSE212277"
)

sample_inventory <- bind_rows(
  lapply(datasets, find_quant_files)
)

sample_inventory

dataset,sample,quant_path,file_exists
<chr>,<chr>,<chr>,<lgl>
GSE146680,infected_mac_1,/standard/harrislab/capstone_2026/quants/GSE146680/infected_mac_1_quant/quant.sf,TRUE
GSE146680,infected_mac_2,/standard/harrislab/capstone_2026/quants/GSE146680/infected_mac_2_quant/quant.sf,TRUE
GSE146680,infected_mac_3,/standard/harrislab/capstone_2026/quants/GSE146680/infected_mac_3_quant/quant.sf,TRUE
GSE146680,infected_mac_4,/standard/harrislab/capstone_2026/quants/GSE146680/infected_mac_4_quant/quant.sf,TRUE
GSE146680,infected_mg_1,/standard/harrislab/capstone_2026/quants/GSE146680/infected_mg_1_quant/quant.sf,TRUE
GSE146680,infected_mg_2,/standard/harrislab/capstone_2026/quants/GSE146680/infected_mg_2_quant/quant.sf,TRUE
GSE146680,infected_mg_3,/standard/harrislab/capstone_2026/quants/GSE146680/infected_mg_3_quant/quant.sf,TRUE
GSE146680,infected_mg_4,/standard/harrislab/capstone_2026/quants/GSE146680/infected_mg_4_quant/quant.sf,TRUE
GSE203655,R1,/standard/harrislab/capstone_2026/quants/GSE203655/R1_quant/quant.sf,TRUE


In [6]:
# Validate the Inventory

cat("Total samples:", nrow(sample_inventory), "\n")
cat("Missing quant.sf files:", sum(!sample_inventory$file_exists), "\n")

print(
  as.data.frame(
    table(sample_inventory$dataset),
    stringsAsFactors = FALSE
  ),
  row.names = FALSE
)

stopifnot(
  nrow(sample_inventory) == 32,
  all(sample_inventory$file_exists)
)

Total samples: 32 
Missing quant.sf files: 0 
      Var1 Freq
 GSE146680    8
 GSE203655    8
 GSE212277   16


In [7]:
# Build the Master Sample Table

build_master_sample_table <- function(base_dir, sample_inventory) {

  meta <- bind_rows(
    read_gse146680_metadata(base_dir),
    read_gse203655_metadata(base_dir),
    read_gse212277_metadata(base_dir)
  )

  meta <- meta %>%
    select(-any_of(c("quant_path", "file_exists")))

  master <- sample_inventory %>%
    left_join(
      meta,
      by = c("dataset", "sample")
    ) %>%
    standardize_cols()

  return(master)
}

In [8]:
master_sample_table <- build_master_sample_table(
  base_dir = base_dir,
  sample_inventory = sample_inventory
)

cat("Master Sample Table created.\n")
cat("Rows:", nrow(master_sample_table), "\n")
cat("Columns:", ncol(master_sample_table), "\n")

New names:
• `` -> `...1`
• `` -> `...2`
• `` -> `...3`
• `` -> `...4`
• `` -> `...5`
• `` -> `...6`
• `` -> `...7`
• `` -> `...8`
• `` -> `...9`
New names:
• `` -> `...1`
• `` -> `...2`
• `` -> `...3`
• `` -> `...4`
• `` -> `...5`
• `` -> `...6`
• `` -> `...7`
• `` -> `...8`
• `` -> `...9`
• `` -> `...10`
• `` -> `...11`
• `` -> `...12`
• `` -> `...13`
• `` -> `...14`
• `` -> `...15`
• `` -> `...16`
• `` -> `...17`
• `` -> `...18`
New names:
• `processed data file` -> `processed data file...13`
• `processed data file` -> `processed data file...14`
• `raw file` -> `raw file...15`
• `raw file` -> `raw file...16`
• `raw file` -> `raw file...17`
• `raw file` -> `raw file...18`


Master Sample Table created.
Rows: 32 
Columns: 24 


In [9]:
# Validate Required Fields
required_fields <- c(
  "dataset",
  "sample",
  "quant_path",
  "file_exists",
  "cell_type",
  "condition",
  "genotype",
  "genotype_short",
  "comparison_group"
)

missing_required_fields <- setdiff(
  required_fields,
  names(master_sample_table)
)

if (length(missing_required_fields) > 0) {
  stop(
    "Missing required columns: ",
    paste(missing_required_fields, collapse = ", ")
  )
}

completeness_check <- data.frame(
  Field = required_fields,
  Missing = vapply(
    required_fields,
    function(field) {
      values <- master_sample_table[[field]]

      sum(
        is.na(values) |
        trimws(as.character(values)) == ""
      )
    },
    numeric(1)
  ),
  stringsAsFactors = FALSE
)

completeness_check

,Field,Missing
,<chr>,<dbl>
dataset,dataset,0
sample,sample,0
quant_path,quant_path,0
file_exists,file_exists,0
cell_type,cell_type,0
condition,condition,0
genotype,genotype,0
genotype_short,genotype_short,0
comparison_group,comparison_group,0


In [10]:
# Validate joins and file paths

duplicate_keys <- duplicated(
  paste(
    master_sample_table$dataset,
    master_sample_table$sample,
    sep = "::"
  )
)

cat("Duplicate dataset/sample keys:", sum(duplicate_keys), "\n")
cat("Missing quant paths:", sum(is.na(master_sample_table$quant_path)), "\n")
cat(
  "Existing quant files:",
  sum(file.exists(master_sample_table$quant_path)),
  "\n"
)

stopifnot(
  nrow(master_sample_table) == 32,
  sum(duplicate_keys) == 0,
  sum(is.na(master_sample_table$quant_path)) == 0,
  all(file.exists(master_sample_table$quant_path))
)

Duplicate dataset/sample keys: 0 
Missing quant paths: 0 
Existing quant files: 32 


In [11]:
# Review Group Distributions

cat("Samples by dataset:\n")
print(table(master_sample_table$dataset))

cat("\nStandardized genotypes by dataset:\n")
print(
  table(
    master_sample_table$dataset,
    master_sample_table$genotype_short,
    useNA = "ifany"
  )
)

cat("\nComparison groups by dataset:\n")
print(
  table(
    master_sample_table$dataset,
    master_sample_table$comparison_group,
    useNA = "ifany"
  )
)

Samples by dataset:

GSE146680 GSE203655 GSE212277 
        8         8        16 

Standardized genotypes by dataset:
           
            5xFAD 5xFAD_Syk_altered Control STAT1_KO Syk_altered WT
  GSE146680     0                 0       8        0           0  0
  GSE203655     0                 0       0        3           0  5
  GSE212277     4                 4       0        0           4  4

Comparison groups by dataset:
           
            Control Disease Disease_Syk_altered Knockout Macrophage Microglia
  GSE146680       0       0                   0        0          4         4
  GSE203655       5       0                   0        3          0         0
  GSE212277       4       4                   4        0          0         0
           
            Syk_altered
  GSE146680           0
  GSE203655           0
  GSE212277           4


In [12]:
# Save Outputs

master_sample_path <- file.path(
  metadata_results_dir,
  "master_sample_table.csv"
)

inventory_path <- file.path(
  metadata_results_dir,
  "sample_inventory.csv"
)

write.csv(
  master_sample_table,
  master_sample_path,
  row.names = FALSE
)

write.csv(
  sample_inventory,
  inventory_path,
  row.names = FALSE
)

cat("Saved Master Sample Table to:\n", master_sample_path, "\n\n")
cat("Saved Sample Inventory to:\n", inventory_path, "\n")

Saved Master Sample Table to:
 /home/kvq4nvn/MSDS/capstone_project/results/metadata/master_sample_table.csv 

Saved Sample Inventory to:
 /home/kvq4nvn/MSDS/capstone_project/results/metadata/sample_inventory.csv 


In [13]:
# Final Summary

cat("============================================================\n")
cat(" MASTER SAMPLE TABLE BUILD SUMMARY\n")
cat("============================================================\n\n")

cat("Datasets:", length(unique(master_sample_table$dataset)), "\n")
cat("Samples:", nrow(master_sample_table), "\n")
cat("Quant files found:", sum(file.exists(master_sample_table$quant_path)), "\n")
cat("Missing required metadata values:", sum(completeness_check$Missing), "\n")
cat("Duplicate sample keys:", sum(duplicate_keys), "\n\n")

cat("Output created:\n")
cat(master_sample_path, "\n\n")

cat("STATUS: MASTER SAMPLE TABLE READY\n")
cat("Next notebook: 02_tximport.ipynb\n")

 MASTER SAMPLE TABLE BUILD SUMMARY

Datasets: 3 
Samples: 32 
Quant files found: 32 
Missing required metadata values: 0 
Duplicate sample keys: 0 

Output created:
/home/kvq4nvn/MSDS/capstone_project/results/metadata/master_sample_table.csv 

STATUS: MASTER SAMPLE TABLE READY
Next notebook: 02_tximport.ipynb


In [15]:
# OPTIONAL INDEPENDENT OUTPUT VALIDATION
#
# This cell rereads the saved CSV files from disk and validates them
# independently of the objects created earlier in this notebook.

master_sample_path <- file.path(
  project_dir,
  "results",
  "metadata",
  "master_sample_table.csv"
)

inventory_path <- file.path(
  project_dir,
  "results",
  "metadata",
  "sample_inventory.csv"
)

validation_results <- list()

# 1. Confirm output files were written.
validation_results$master_file_exists <- file.exists(master_sample_path)
validation_results$inventory_file_exists <- file.exists(inventory_path)

if (!validation_results$master_file_exists) {
  stop("Saved Master Sample Table was not found: ", master_sample_path)
}

if (!validation_results$inventory_file_exists) {
  stop("Saved Sample Inventory was not found: ", inventory_path)
}

# 2. Reread outputs from disk.
master_check <- read.csv(
  master_sample_path,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

inventory_check <- read.csv(
  inventory_path,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

# 3. Validate expected dimensions.
validation_results$master_has_32_rows <- nrow(master_check) == 32
validation_results$inventory_has_32_rows <- nrow(inventory_check) == 32
validation_results$has_3_datasets <-
  length(unique(master_check$dataset)) == 3

# 4. Validate required columns.
required_columns <- c(
  "dataset",
  "sample",
  "quant_path",
  "file_exists",
  "cell_type",
  "condition",
  "genotype",
  "genotype_short",
  "comparison_group"
)

validation_results$required_columns_present <-
  all(required_columns %in% names(master_check))

# 5. Validate required-field completeness.
required_values <- master_check[, required_columns, drop = FALSE]

validation_results$required_fields_complete <- all(
  vapply(
    required_values,
    function(x) {
      !any(is.na(x) | trimws(as.character(x)) == "")
    },
    logical(1)
  )
)

# 6. Validate unique sample keys.
sample_keys <- paste(
  master_check$dataset,
  master_check$sample,
  sep = "::"
)

validation_results$sample_keys_unique <- !anyDuplicated(sample_keys)

# 7. Validate quantification files using current filesystem state.
quant_files_exist <- file.exists(master_check$quant_path)

validation_results$all_quant_files_exist <- all(quant_files_exist)
validation_results$quant_file_count <- sum(quant_files_exist)

# 8. Validate that the saved registry matches the saved inventory.
inventory_keys <- paste(
  inventory_check$dataset,
  inventory_check$sample,
  sep = "::"
)

validation_results$inventory_keys_match <- setequal(
  sample_keys,
  inventory_keys
)


# 9. Confirm expected sample counts for the current three datasets.
# FUTURE DATASETS:
# Update this section when adding a new GEO study.

expected_dataset_sample_counts <- c(
  "GSE146680" = 8,
  "GSE203655" = 8,
  "GSE212277" = 16
)

observed_counts <- table(master_check$dataset)

validation_results$sample_counts_correct <- all(
  names(expected_dataset_sample_counts) %in% names(observed_counts)
) && all(
  observed_counts[names(expected_dataset_sample_counts)] == expected_counts
)


# 10. Display validation report.
validation_report <- data.frame(
  Check = c(
    "Master Sample Table file exists",
    "Sample Inventory file exists",
    "Master Sample Table has 32 rows",
    "Sample Inventory has 32 rows",
    "Three datasets are represented",
    "Required columns are present",
    "Required fields are complete",
    "Dataset/sample keys are unique",
    "All quant.sf files exist",
    "Saved inventory and registry keys match",
    "Dataset sample counts are correct"
  ),
  Passed = c(
    validation_results$master_file_exists,
    validation_results$inventory_file_exists,
    validation_results$master_has_32_rows,
    validation_results$inventory_has_32_rows,
    validation_results$has_3_datasets,
    validation_results$required_columns_present,
    validation_results$required_fields_complete,
    validation_results$sample_keys_unique,
    validation_results$all_quant_files_exist,
    validation_results$inventory_keys_match,
    validation_results$sample_counts_correct
  ),
  stringsAsFactors = FALSE
)

validation_report$Status <- ifelse(
  validation_report$Passed,
  "PASS",
  "FAIL"
)

cat("============================================================\n")
cat(" INDEPENDENT OUTPUT VALIDATION\n")
cat("============================================================\n\n")

print(
  validation_report[, c("Check", "Status")],
  row.names = FALSE
)

cat("\nQuant files verified:", validation_results$quant_file_count, "\n")

if (all(validation_report$Passed)) {
  cat("\n✓ ALL MASTER SAMPLE TABLE CHECKS PASSED\n")
} else {
  failed_checks <- validation_report$Check[
    !validation_report$Passed
  ]

  cat("\n✗ VALIDATION FAILED\n")
  cat(
    "Failed check(s):\n- ",
    paste(failed_checks, collapse = "\n- "),
    "\n",
    sep = ""
  )

  stop("Resolve failed validation checks before continuing.")
}

 INDEPENDENT OUTPUT VALIDATION

                                   Check Status
         Master Sample Table file exists   PASS
            Sample Inventory file exists   PASS
         Master Sample Table has 32 rows   PASS
            Sample Inventory has 32 rows   PASS
          Three datasets are represented   PASS
            Required columns are present   PASS
            Required fields are complete   PASS
          Dataset/sample keys are unique   PASS
                All quant.sf files exist   PASS
 Saved inventory and registry keys match   PASS
       Dataset sample counts are correct   PASS

Quant files verified: 32 

✓ ALL MASTER SAMPLE TABLE CHECKS PASSED
